# 05 – Data Merge: Silver → Gold

Joins silver tables into the gold layer:
- Ratings + Books + Users (inner join on ISBN and user_id)
- Pre-computed aggregation tables: `gold.books_summary`, `gold.authors_summary`
- Only explicit ratings (rating > 0) are used

**Source:** `silver.{books, users, ratings}`
**Target:** `gold.{book_ratings, books_summary, authors_summary}`
**Config:** `config/05_data_merge.json`

In [0]:
import json, sys, importlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc/config/00_setup.json"
with open(SETUP_CONFIG) as f:
    setup = json.load(f)

sys.path.insert(0, setup["project_path"])

for mod in [m for m in list(sys.modules.keys()) if m.startswith("source")]:
    del sys.modules[mod]

DataMerge = importlib.import_module("source.05_data_merge").DataMerge

CONFIG_PATH = f"{setup['project_path']}/config/05_data_merge.json"

In [0]:
pipeline = DataMerge(config_path=CONFIG_PATH)
df = pipeline.run()

In [0]:
display(df)

In [0]:
from pyspark.sql import functions as F

print(f"New table: \ngold.book_ratings is a single denormalised fact table with all columns from all three sources.")
print(f"Total rated records: {df.count():,}")
print(f"Unique books: {df.select('isbn').distinct().count():,}")
print(f"Unique users: {df.select('user_id').distinct().count():,}")
print(f"Unique authors: {df.select('author').distinct().count():,}")
print(f"Rating range: {df.agg(F.min('book_rating')).collect()[0][0]} - {df.agg(F.max('book_rating')).collect()[0][0]}")